In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
  Using cached qiskit-aer-0.15.1.tar.gz (6.6 MB)
  Installing build dependencidone
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for qiskit-aer (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [156 lines of output]
      /private/var/folders/cq/zydnrrq917xbxw2wbll_sz0w0000gn/T/pip-build-env-74g8vl9m/overlay/lib/python3.13/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/

In [6]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

root2 = math.sqrt(2)
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2)

b0_matrix = [ [-1/denom1,(1+root2)/denom1],
              [ 1/denom2,(root2-1)/denom2] ]
b1_matrix = [ [ 1/denom1,(1+root2)/denom1],
              [-1/denom2,(root2-1)/denom2] ]
b0_transform = Operator(b0_matrix)
b1_transform = Operator(b1_matrix)

n = 100

def create_state():
    q = QuantumCircuit(2)
    q.h(0)
    return q

def quantum_sim(c, n):
    qc = create_state()
    qc.measure_all()
    qc_sim = BasicSimulator()
    qc_compiled = transpile(qc, qc_sim)
    qc_job = qc_sim.run(qc_compiled, shots=n)
    qc_result = qc_job.result()
    qc_counts = qc_result.get_counts(qc_compiled)

    qc00 = qc_counts.get("00",0)
    qc01 = qc_counts.get("01",0)
    qc10 = qc_counts.get("10",0)
    qc11 = qc_counts.get("11",0)
    
    sim = BasicSimulator()
    compiled = transpile(c, sim)
    job = sim.run(compiled, shots=n)
    result = job.result()
    counts = result.get_counts(compiled)
    
    c00 = counts.get("00",0)
    c01 = counts.get("01",0)
    c10 = counts.get("10",0)
    c11 = counts.get("11",0)

    average = (c00 - c01 - c10 + c11) / n
    key_len = qc00 + qc01 + qc10 + qc11
    err_rate = (qc01 - qc10) / key_len if key_len > 0 else 0
    
    return average, key_len, err_rate

def run(circuit, matrix):
    circuit.unitary(matrix,[1])
    circuit.measure_all()
    average, key_len, err_rate = quantum_sim(circuit, n)
    return average, key_len, err_rate

c_a0b0 = create_state()
c_a0b1 = create_state()
c_a1b0 = create_state()
c_a1b1 = create_state()

a0b0_avg, key_len, err_rate = run(c_a0b0, b0_matrix)
a0b1_avg, key_len, err_rate = run(c_a0b1, b1_matrix)
a1b0_avg, key_len, err_rate = run(c_a1b0, b0_matrix)
a1b1_avg, key_len, err_rate = run(c_a1b1, b1_matrix)

bell_param = abs(a0b0_avg + a0b1_avg + a1b0_avg + a1b1_avg)
print("Bell parameter:", bell_param)
print("Key length:", key_len)
print("Error rate:", err_rate)

Bell parameter: 0.52
Key length: 100
Error rate: 0.45
